# An Implementation of safPAKE (Crypto2025) #

## This Notebook Tests the Practicality of safPAKE ##

In [1]:
import face_recognition
import matplotlib.pyplot as plt
import numpy as np
import random
from tqdm import tqdm
import json
import time

### First We Define the CosineLSH coding functionality

In [2]:
class CosineLSH:

    """
    Reduction_matrix: the first pre-processing step that reduce the input dimension for a facial (128) to a smaller one (target_dimension)
    """

    def __init__(self, hashed_bits, dimension, seed=None):

        #---------------- Avoid High Dimensionality, use JL Lemma to reduce dimension----------------
        rng = np.random.default_rng(seed)

        #-------------------------- this part defines all the standard LSH mechanism -------------------
        self.num_bits = hashed_bits
        self.dimension = dimension
        # Generate random hyperplanes
        self.hyperplanes = rng.standard_normal(size=(hashed_bits, dimension))

        norms = np.linalg.norm(self.hyperplanes, axis=1, keepdims=True)
        
        # Avoid division by zero (unlikely with Gaussian, but good practice)
        norms[norms == 0] = 1 
        
        self.hyperplanes = self.hyperplanes / norms

        self.filtered_indices=None
    
    @staticmethod
    def to_fixed_width(n, bit_width):
        # Create a mask of all 1s for the desired width
        mask = (1 << bit_width) - 1
        # Force the number into that width
        return n & mask

    def hash(self, vector):
        # Project the vector onto each hyperplane
        projections = np.dot(self.hyperplanes, vector)
        # Determine the sign of each projection
        signs = projections >= 0
        # Convert boolean array to integer hash code
        hash_code = 0
        for bit in signs:
            hash_code = (hash_code << 1) | int(bit)
            
        return self.to_fixed_width(int(hash_code),self.num_bits)
    
    def filter_by_hyperplane(self, vector, num_vec): 
            """
            Unlike filter_out_bad_ones (which uses an angle threshold), this function 
            discards a fixed number (num_vec) of hyperplanes that are most perpendicular 
            to the input vector.
            """
            # 1. Normalize input vector
            norm_vec = np.linalg.norm(vector)
            if norm_vec == 0:
                return np.array([], dtype=int)
            
            unit_vec = vector / norm_vec

            # 2. Calculate Cosine (Dot Product)
            # Result is array of cosines between -1 and 1
            cos_theta = np.dot(self.hyperplanes, unit_vec)
            
            # 3. Sort by "Perpendicular-ness"
            # We want values closest to 0. Taking abs() makes 0 the minimum.
            abs_cos = np.abs(cos_theta)
            
            # np.argsort returns the indices that would sort the array from smallest to largest.
            # The smallest values in abs_cos are the ones closest to 0 (most perpendicular).
            sorted_indices = np.argsort(abs_cos)
            
            # 4. Select the top k indices
            bad_indices = sorted_indices[:num_vec]

            # 5. Update state
            self.filtered_indices = bad_indices
            
            return bad_indices

    def filter_out_bad_ones(self, vector, degree_tolerance):
        # 1. Normalize input vector (High Dim)
        norm_vec = np.linalg.norm(vector)
        if norm_vec == 0:
            return np.array([], dtype=int)
        
        unit_vec = vector / norm_vec
    
        # 4. Calculate Cosine (Dot Product)
        # Now this is a valid cosine between -1 and 1
        cos_theta = np.dot(self.hyperplanes, unit_vec)
    
        # 5. Define Threshold
        threshold = np.sin(np.radians(degree_tolerance))
        
        # 6. Filter
        bad_indices = np.where(np.abs(cos_theta) < threshold)[0]

        self.filtered_indices=bad_indices
        
        return bad_indices

    def filtered_hamming_weights(self, intA: int, intB: int) -> int:
        if type(self.filtered_indices)==None: raise AssertionError("No existing Filtering")

        # 1. Calculate Standard XOR
        xor_result = intA ^ intB

        ignore_mask = 0
        shifts = (self.num_bits - 1) - self.filtered_indices
        for shift in shifts:
            ignore_mask |= (1 << int(shift))

        full_mask = (1 << self.num_bits) - 1
        clean_inverse_mask = (~ignore_mask) & full_mask
        
        filtered_xor = xor_result & clean_inverse_mask
        
        return bin(filtered_xor).count('1')

    def hamming_distance(self, a: int, b: int) -> int:
        return bin(a ^ b).count('1')

### Now we define the image to encoding functionality for an particular image ###

In [3]:
def images_to_encoding(path):
    image = face_recognition.load_image_file(path)
    
    if(len(face_recognition.face_encodings(image))==0): 
        return [None]

    if(len(face_recognition.face_encodings(image)[0])!=0):
        return face_recognition.face_encodings(image)[0]

    return [None]


def rand_select(avoidance=None):
    if avoidance==None: 
        return random.randint(0,100)
    else:
        selected=random.randint(0,100)
        while selected==avoidance:
            selected=random.randint(0,100)
        return selected

## MAJOR PART: OPRF IMPLEMENTATION ##

### Code below implements OPRF in OOP module ###

In [4]:
import ctypes
import ctypes.util
import hashlib
import os
from typing import Tuple

# Constants (libsodium uses 32-byte encodings for scalars/points, 64-byte hash-to-group)
SCALAR_LEN = 32
POINT_LEN = 32
HASH_TO_POINT_BYTES = 64

1. We load the libsodium for EC operations

In [5]:
def load_sodium():
    # Try common names
    for name in ("sodium", "libsodium"):
        path = ctypes.util.find_library(name)
        if path:
            try:
                return ctypes.CDLL(path)
            except Exception:
                pass
    # If not found via find_library, try to load from CONDA_PREFIX
    conda_prefix = os.environ.get("CONDA_PREFIX")
    if conda_prefix:
        candidates = [
            os.path.join(conda_prefix, "Library", "bin", "libsodium.dll"),  # common on Windows
            os.path.join(conda_prefix, "Library", "bin", "sodium.dll"),
            os.path.join(conda_prefix, "bin", "libsodium.dll"),
            os.path.join(conda_prefix, "lib", "libsodium.so"),
        ]
        for c in candidates:
            if os.path.exists(c):
                return ctypes.CDLL(c)
    raise OSError("Could not locate libsodium. Make sure 'libsodium' is installed and on PATH (conda-forge).")

sodium = load_sodium()

2. We Initialize the libsodium for future works

In [6]:
# Initialize libsodium
if hasattr(sodium, "sodium_init"):
    ret = sodium.sodium_init()
    if ret < 0:
        raise RuntimeError("sodium_init failed")

# set arg/return types for functions we will use
# int crypto_core_ristretto255_from_hash(unsigned char *p, const unsigned char *r);
sodium.crypto_core_ristretto255_from_hash.argtypes = [ctypes.c_void_p, ctypes.c_void_p]
sodium.crypto_core_ristretto255_from_hash.restype = ctypes.c_int

# void crypto_core_ristretto255_scalar_random(unsigned char *s);
sodium.crypto_core_ristretto255_scalar_random.argtypes = [ctypes.c_void_p]
sodium.crypto_core_ristretto255_scalar_random.restype = None

# int crypto_scalarmult_ristretto255(unsigned char *q, const unsigned char *n, const unsigned char *p);
sodium.crypto_scalarmult_ristretto255.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
sodium.crypto_scalarmult_ristretto255.restype = ctypes.c_int

# int crypto_core_ristretto255_scalar_invert(unsigned char *out, const unsigned char *s);
sodium.crypto_core_ristretto255_scalar_invert.argtypes = [ctypes.c_void_p, ctypes.c_void_p]
sodium.crypto_core_ristretto255_scalar_invert.restype = ctypes.c_int

3. Define all the necessary helper functions 

In [7]:
# Helpers
def random_scalar() -> bytes:
    out = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_random(ctypes.byref(out))
    return bytes(bytearray(out))

def hash_to_point(msg: bytes) -> bytes:
    # SHA-512 -> 64 bytes input to from_hash
    h = hashlib.sha512(msg).digest()
    assert len(h) == HASH_TO_POINT_BYTES
    out = (ctypes.c_ubyte * POINT_LEN)()
    r = sodium.crypto_core_ristretto255_from_hash(ctypes.byref(out), ctypes.c_char_p(h))
    if r != 0:
        raise RuntimeError("crypto_core_ristretto255_from_hash failed")
    return bytes(bytearray(out))

def scalarmult(scalar: bytes, point: bytes) -> bytes:
    assert len(scalar) == SCALAR_LEN and len(point) == POINT_LEN
    out = (ctypes.c_ubyte * POINT_LEN)()
    r = sodium.crypto_scalarmult_ristretto255(ctypes.byref(out),
                                              ctypes.c_char_p(scalar),
                                              ctypes.c_char_p(point))
    if r != 0:
        raise RuntimeError("crypto_scalarmult_ristretto255 failed (maybe identity)")
    return bytes(bytearray(out))

def scalar_invert(scalar: bytes) -> bytes:
    assert len(scalar) == SCALAR_LEN
    out = (ctypes.c_ubyte * SCALAR_LEN)()
    r = sodium.crypto_core_ristretto255_scalar_invert(ctypes.byref(out), ctypes.c_char_p(scalar))
    if r != 0:
        raise RuntimeError("scalar invert failed")
    return bytes(bytearray(out))

# Simple deterministic KDF for PRF output (use HKDF or SHA512 as needed)
def prf_final(point_bytes: bytes) -> bytes:
    return hashlib.sha512(b"OPRF-v1" + point_bytes).digest()

In [8]:
class OPRFServer:
    def __init__(self):
        # 1. The paper defines k as a random bitstring {0,1}^kappa [cite: 172]
        # We generate 32 bytes (256 bits) for the key.
        self.key = os.urandom(32)

    def evaluate(self, blinded_point: bytes) -> bytes:
        # 2. Apply H3 to map the key string to a scalar [cite: 170, 179]
        # b := a ^ H3(k)
        # Since we must use provided helpers, we use hashlib to generate the bytes
        # and take the first 32 bytes to form a valid scalar input for scalarmult.
        # (This acts as the H3 oracle).
        h3_digest = hashlib.sha512(self.key).digest()
        scalar_h3 = h3_digest[:32] 
        
        # 3. Compute b = a * H3(k) (scalarmult is additive notation for exponentiation)
        return scalarmult(scalar_h3, blinded_point)


class OPRFClient:
    def __init__(self):
        pass

    def blind(self, inp: bytes) -> Tuple[bytes, Tuple[bytes, bytes]]:
        # 1. Matches H1(x) -> hash_to_point(x) 
        P = hash_to_point(inp)
        
        # 2. Sample random scalar r 
        r = random_scalar()
        
        # 3. Compute a := H1(x)^r -> scalarmult(r, P) 
        blinded = scalarmult(r, P)
        
        # Return blinded element and state.
        # NOTE: We MUST save 'inp' (x) in the state to compute H2(x, ...) later.
        return blinded, (r, inp)

    def finalize(self, client_state: Tuple[bytes, bytes], evaluated: bytes) -> bytes:
        # Unpack state
        r, inp = client_state
        
        # 1. Compute 1/r 
        r_inv = scalar_invert(r)
        
        # 2. Unblind: b^(1/r) -> scalarmult(r_inv, evaluated) 
        unblinded = scalarmult(r_inv, evaluated)
        
        # 3. Matches H2(x, unblinded) 
        # The paper requires hashing BOTH the input x and the unblinded element.
        # We construct the input to H2 as (inp || unblinded).
        h2_input = inp + unblinded
        
        # We use SHA-512 for H2 as implied by the use of sha512 in other helpers.
        return hashlib.sha512(h2_input).digest()

### A Quick Demo of the OPRF ###

It OPRF the string "Hello World"

In [9]:
server = OPRFServer()
client = OPRFClient()
cli=OPRFClient()

In [10]:
msg = b"hello world"
blinded, state = client.blind(msg)
blinded2, state2=cli.blind(msg)
evaluated = server.evaluate(blinded)
eval=server.evaluate(blinded2)
out1 = client.finalize(state, evaluated)
out2 =client.finalize(state2,eval)
print("OPRF output:", out1.hex())
print("OPRF output:", out2.hex())

OPRF output: 08c772c30baf39cbdb6e2f3effe0c25d30412e6e30932e0e7b703553bab20e6845f9ce3e01ad91eeacaad84edb31b27cf65923618a2482f8d6003d5cfe643f6a
OPRF output: 08c772c30baf39cbdb6e2f3effe0c25d30412e6e30932e0e7b703553bab20e6845f9ce3e01ad91eeacaad84edb31b27cf65923618a2482f8d6003d5cfe643f6a


## Now we also gonna need to define the AEAD (AES) algorithm ###

In [11]:
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.backends import default_backend

In [12]:
def derive_aes_key(ek: bytes) -> bytes:
    """Derives a 256-bit AES key from the OPRF output's ek using HKDF."""
    # Use a fixed salt and info string for deterministic key derivation
    # The key ek must be K bytes (32 bytes) long.
    salt = b'safpake_key_salt'
    info = b'safpake_aes_key_derivation'
    
    # HKDF is used to expand the PRF key 'ek' into a suitable 256-bit AES key
    return HKDF(
        algorithm=hashes.SHA256(),
        length=16, # AES-256 requires a 32-byte key
        salt=salt,
        info=info,
        backend=default_backend()
    ).derive(ek)

class AuthenticatedEncryption:
    """
    Implements the Authenticated Encryption scheme E (Enc, Dec)
    required for safPAKE using AES-256-GCM.
    """

    @staticmethod
    def Encrypt(ek, data: bytes) -> bytes:
        """
        Encrypts data using AES-ECB.
        
        Args:
            data (bytes): The plaintext data to encrypt.
            
        Returns:
            bytes: The encrypted ciphertext.
        """
        key=derive_aes_key(ek)

        # We create a new cipher object for every operation to reset state
        cipher = AES.new(key, AES.MODE_ECB)
        
        # Pad the data to be a multiple of the block size (16 bytes)
        # standard PKCS7 padding is used here.
        padded_data = pad(data, AES.block_size)
        
        return cipher.encrypt(padded_data)

    @staticmethod
    def Decrypt(ek, ciphertext: bytes) -> bytes:
        """
        Decrypts data using AES-ECB.
        
        Args:
            ciphertext (bytes): The encrypted data to decrypt.
            
        Returns:
            bytes: The original plaintext.
        """
        key=derive_aes_key(ek)

        cipher = AES.new(key, AES.MODE_ECB)
        
        # Decrypt and then remove the padding
        padded_plaintext = cipher.decrypt(ciphertext)
        plaintext = unpad(padded_plaintext, AES.block_size)
        
        return plaintext
     

# Now We need to implement the safPAKE for FAKE #

In [13]:
class UserProfile:
    """
    This is the user profile within a server, it does not have a default init but only init everything when register

    For each user: it should be associated with the following (from JIAYU's work):

        1. OPRF key: as an OPRF Server class 
        2. Error_ball DB: it's a tag -> value so it's a dictionary
        3. User's Name: a string 
    """

    def __init__(self,hyperplane,tolerance):

        #Create the user DB
        self.DB=dict()

        #Create the locality hashing thing
        self.LSH=CosineLSH(hyperplane,128)

        # Tolerance is no longer in degree but in number of hypervectors
        self.tolerance=tolerance

        #select the OPRF key
        self.oprf_s=OPRFServer()

        # the server operation needs to be blinded first
        self.oprf_c=OPRFClient()

        #create the AES functionality
        self.AE=AuthenticatedEncryption()

        #save the base_face for client's re-construction
        #Please note that this should be in the ciphertext rather than here, it's mainly for simplicity of demostration
        self.registered_face=None
     

    def derive_variants(self, orig_face_vec, pw): # this function contains a bug
        """
        Given the base password (as a vector), the hyperplane, and the degree of tolerance, derive all possible variants of the pw in byte string form

        1. Since the pw is already in bit form as an integer, I will use hash's derive ignored bit functionality to get all the bits
        2. Then from these ignored bits, I get all the possible combinations of these and return them as integers. 
        """

        self.registered_face=orig_face_vec

        indices=self.LSH.filter_by_hyperplane(orig_face_vec,self.tolerance)

        shifts = [int((self.LSH.num_bits - 1) - idx) for idx in indices]

        # 2. Prepare the template: 
        # Clear any bits at the target positions and truncate to num_bits
        full_mask = (1 << self.LSH.num_bits) - 1
        base_template = int(pw) & full_mask

        for shift in shifts:
            base_template &= ~(1 << shift)
            
        variants = []
        num_combinations = 1 << len(indices)

        # 3. Iterate through all combinations (binary counter method)
        for i in range(num_combinations):
            current_variant = base_template
            for j, shift in enumerate(shifts):
                # If the j-th bit of our counter is 1, set the bit in our variant
                if (i >> j) & 1:
                    current_variant |= (1 << shift)
            
            variants.append(int(current_variant))

        return variants


    def register(self,face): 
        # NOTE the pw here is the result of the hashed string of the face
        """
        From the pw (base face, an 128 np array), the server should obtain the following:

        1. all variations of the pw
        2. each variation -----> tag, ek 
        3. put them into database entries of the self.DB
        """

        # 1. First we hash the face vector and project into a bit_string (this is an integer) 
        base_pw=self.LSH.hash(face)

        # start_derive = time.time()
        
        # 2. Then we derive the error ball from the base password
        all_variants=self.derive_variants(face,base_pw)

        # end_derive = time.time()
        # print(f"[Timing] Variant Derivation ({len(all_variants)} variants): {end_derive - start_derive:.4f} seconds")

        # 3. Finally we fill the DB with these entries
        #       - For each one in variant, derive tag, key via OPRF
        #       - Then put the dictionary entry 

        # At outside, we prepare the ctxt_msg= (base_pw || oprf_key)
        #   1. Change the base_pw to bytes
        #   2. Concatenate it with the oprf key 
        pw_base_bytes = base_pw.to_bytes(self.LSH.num_bits, byteorder='big')

        ctxt_msg= pw_base_bytes + self.oprf_s.key

        # print("Variants Derivation Done:\n Creating Database ...")

        # start_db = time.time()

        for pw_prime in all_variants:

            # convert the pw_prime into byte format for oprf to actually eval
            pw_prime_bytes=pw_prime.to_bytes(self.LSH.num_bits, byteorder='big')

            blinded,state=self.oprf_c.blind(pw_prime_bytes)

            # derive the tag, note that ek can be derived directly from AuthenticatedEncryption class

            temp_res=self.oprf_s.evaluate(blinded)

            tag=self.oprf_c.finalize(state,temp_res)

            ctxt=self.AE.Encrypt(tag,ctxt_msg)

            # because the oprf takes in the state, we save it otherwise it outputs something very different next time
            self.DB[tag]=ctxt

        # end_db = time.time()
        # print(f"[Timing] Database Population: {end_db - start_db:.4f} seconds")
        # print(f"[Timing] Total Registration Time: {end_db - start_derive:.4f} seconds")

        
#--------------------------The following performs the authentication steps--------------------------------
# 1. The server first ready for the client to start the oprf (waiting for the OPRF and send it)
# 2. The server sends the whole database along with the OPRF result 

#### NOTE: step 1 and 2 are in the same step
#### There should also be an opaque step, but we will ignore that for here

    def authenticate(self, blinded_oprf, state, client):
        """
        The second argument 'client' also have the verify authentication API that takes in
        1. OPRF server's output
        2. the final database (in dictionary form)
        """

        oprf_output=self.oprf_s.evaluate(blinded_oprf)

        return client.verify(oprf_output,state, # transmit back the OPRF data (happens in actual case)
                             self.DB, # transmit back the DB (happens in actual case)
                             self.LSH, #transmit the LSH for hashing
                             self.tolerance, # transmit back the public information for hashing and error-ball re-construction (happens in actual case)
                             self.registered_face # transmit back some aux information (this is not the base in actual), face should be stored in the DB after encryption
                            )

### 1. We gonna do the server first ###

In [14]:
class Server:
    """
    In the initialization function, the parameters are as follows:
    1. K: the length of the key


    It should perform the following functionalities:
        - Establish a user profile-> in form of a dictionary with associated user name
    """
    def __init__(self,K=32, tolerance=10, disjoint_bagging=1):

        self.all_users=dict() 

        self.AES_key_len=K

        self.tolerance=tolerance

        self.bags=disjoint_bagging

    def register(self,user_name:str,face,hyperplane=66):
        """
        User will register a face in 128 dimention np.array 

        1. create a new user profile and also specify the tolerance for the profile
        2. The tolerance parameter here is the degree of variance from the base_face
        3. Hyperplane parameter is the number of bits to be eventually hashed
        """

        oprf_key=os.urandom(32)

        profiles=[UserProfile(hyperplane, self.tolerance) for _ in range(self.bags)]

        for each_profile in profiles:
            each_profile.oprf_s.key=oprf_key
            each_profile.register(face)
    
        #-----------------Once everything is done, put this profile into the All-user DB----------------------------
        self.all_users[user_name]=profiles

        return True
    
    def authenticate(self,user_name:str, client):

        # To authenticate a user, we will first retrieve the user profile and its assocaited information

        # handle the non-existing user case first
        if user_name not in self.all_users: return False

        # Call the underlying authenticate
        user_profile=self.all_users[user_name]

        temp_db=[each_profile.DB for each_profile in user_profile]
        temp_LSH=[each_profile.LSH for each_profile in user_profile]

        client.receive_all_info(temp_LSH,temp_db)

        for each_profile in user_profile:

            oprf_blinded, state=client.get_oprf_output(each_profile.LSH)
            if each_profile.authenticate(oprf_blinded, state, client): return True

        return False

In [15]:
class Client:
    def __init__(self, username:str,face):

        # client will then have its username, but no face stored
        self.name=username

        # we will also give the client access to OPRF_Client functionalities
        self.oprf_client=OPRFClient()

        # Also give it the oprf server functionality to get the final result
        self.oprf_server=OPRFServer()

        # we will also give the client the ability to do the authenticated encryption
        self.AE=AuthenticatedEncryption()

        # just initialize this to 0 first, then check
        self.hashed_bits=0

        self.face=face

        """
        Now I need to have a list that saves all the session DB received for bagging's reconstruction check
        """
        self.temp_hyperplanes=[]
        self.temp_db=[] # just
    
    def get_oprf_output(self, LSH):
        """
        At this step, the client receives the Cosine Hashing Hyperplanes from the server. 
        Then it hashes its face and blind it using the oprf functionality
        """
        
        # this is the number of bits that the lsh hashed into
        self.hashed_bits=LSH.num_bits

        # now we hash the face into a byte for OPRF
        hashed_face=LSH.hash(self.face)

        hashed_face=hashed_face.to_bytes(LSH.num_bits, byteorder="big")

        blinded, state=self.oprf_client.blind(hashed_face)

        # now we return the blinded OPRF output
        return blinded, state
    

    @staticmethod
    def recover_int_and_bytes(decrypted_msg: bytes, hashed_num_bits: int):
        """
        Recovers the integer and byte array from the combined byte array.
        """
        
        # 1. Split the combined array based on the known size.
        # The first 'size' bytes are the integer.
        pw_base_bytes_recovered = decrypted_msg[:hashed_num_bits]
        
        # The remaining bytes are the original key.
        key_recovered = decrypted_msg[hashed_num_bits:]
        
        # 2. Decode the integer bytes back into an int.
        pw_base_recovered = int.from_bytes(pw_base_bytes_recovered, byteorder='big')
        
        return pw_base_recovered, key_recovered
    
    def reconstruction_check(self,tolerance, oprf_key:bytearray, LSH:CosineLSH, DB:dict, registered_face):
        """
        NOTE: the reconstruction step bascially re-creates the whole database and check if the DB is correctly generated
        It basically runs the same registration protocol (again) but check if each DB entry is correctly generated
        """

        # Create the Oprf functionality and force the OPRF-server key to be the key
        oprfs=OPRFServer()
        oprfc=OPRFClient()
        oprfs.key=oprf_key

        # Now 
        indices=LSH.filter_by_hyperplane(registered_face,tolerance)
        pw_base=LSH.hash(registered_face)

        shifts = [int((LSH.num_bits - 1) - idx) for idx in indices]

        # 2. Prepare the template: 
        # Clear any bits at the target positions and truncate to num_bits
        full_mask = (1 << LSH.num_bits) - 1
        base_template = int(pw_base) & full_mask
        for shift in shifts:
            base_template &= ~(1 << shift)
            
        num_combinations = 1 << len(indices)

        # 3. Iterate through all combinations (binary counter method) and check the DB entry
        for i in range(num_combinations):
            current_variant = base_template
            for j, shift in enumerate(shifts):
                # If the j-th bit of our counter is 1, set the bit in our variant
                if (i >> j) & 1:
                    current_variant |= (1 << shift)
            
            blinded,state=oprfc.blind(int(current_variant).to_bytes(LSH.num_bits,"big"))
            tag=oprfc.finalize(state,oprfs.evaluate(blinded))

            if DB.get(tag,0)==0: return False
        
        return True
    
    def receive_all_info(self,LSH_bags,Database_bags):
        self.temp_hyperplanes=LSH_bags
        self.temp_db=Database_bags

    def verify(self, oprf_output:bytearray, state, databse:dict, LSH:CosineLSH, degree_of_tolerance: int, registred_face:np.array):
        # step 0, check if we first send into the blinded thing
        if self.hashed_bits==0: return False

        # 1. first get the tag by unblind the OPRF 
        tag=self.oprf_client.finalize(state, oprf_output)

        # 2. Then see if the entry exists
        ctxt = databse.get(tag, 0) # Returns 0 if key is missing

        if ctxt==0: 
            return False

        else:

            # start_verify = time.time()

            # value here is the ctxt= Enc (ek, (pw || oprf_Key))
            dec_digest=self.AE.Decrypt(tag,ctxt)

            # do the type conversion using the recover static method
            pw_base, oprf_key=self.recover_int_and_bytes(dec_digest,self.hashed_bits)

            # now reconstruct the whole thing along with other bags

            # if not self.reconstruction_check(degree_of_tolerance,oprf_key,LSH,databse,registred_face): return False

            for i in range(len(self.temp_db)):
                if not self.reconstruction_check(degree_of_tolerance,oprf_key,self.temp_hyperplanes[i],self.temp_db[i],registred_face): return False

            # end_verify = time.time()

            # print(f"[Timing] User Verification: {end_verify - start_verify:.4f} seconds")

        return True

# Section 3: Experiment Setup #

1. We do a toy example of a single face
2. We perform test on multiple faces and test false positive/negative

### Get all faces loaded, saved to json file ###

In [16]:
# Just in case we have additional dataset to test on
folder_extracted="train"

# define the official data structure in RAM 

facial_data=dict()

### Now the Databse Building Process ##

1. We iterate through all the folders 

2. For each folder we encode the first CAP images 

3. Store into the DB

In [17]:
def restore(obj):
    if isinstance(obj, list):
        # If it's a list of numbers, convert to numpy array
        if all(isinstance(x, (int, float)) for x in obj):
            return np.array(obj)
        # Otherwise recurse element-wise
        return [restore(x) for x in obj]
    if isinstance(obj, dict):
        return {k: restore(v) for k, v in obj.items()}
    return obj

with open(f"data_{folder_extracted}.json", "r") as f:
    facial_data_loaded = json.load(f)

facial_data= restore(facial_data_loaded)

## Toy example for a single face ##

In [18]:
def mean_faces(encodings):
    arr = np.array(encodings, dtype=float)
    return arr.mean(axis=0)

def rand_select(avoidance=None):
    if avoidance==None: 
        return random.randint(0,100)
    else:
        selected=random.randint(0,100)
        while selected==avoidance:
            selected=random.randint(0,100)
        return selected

In [19]:
# then we define the client and the server
server=Server(tolerance=14,disjoint_bagging=3)

In [ ]:
face_base=mean_faces(facial_data[list(facial_data.keys())[4]])
server.register("test_user",face_base,hyperplane=88)

In [42]:
face_authen=facial_data[list(facial_data.keys())[4]][rand_select()]
client2=Client("test_user",face_authen)
server.authenticate("test_user",client2)

[Timing] User Verification: 20.7014 seconds


True

## Experiment for Actual Time computation ##

In [22]:
import pandas as pd
from tqdm import tqdm

def run_safpake_experiment(facial_data, trials_per_config=10):
    """
    Benchmarks safPAKE by averaging multiple trials per configuration.
    Only successful authentication times are included in the auth average.
    """
    # 1. Define the Parameter Grid
    N_list = [64, 72, 80, 88, 96, 104]
    bags_list = [1, 3, 5, 7, 9, 11, 13]
    tolerance_list = list(range(7, 15))  # 7 through 14
    
    results = []
    user_keys = list(facial_data.keys())
    
    total_configs = len(N_list) * len(bags_list) * len(tolerance_list)
    total_runs = total_configs * trials_per_config
    
    print(f"=== Starting Robust safPAKE Timing Experiment ===")
    print(f"Configurations: {total_configs} | Trials per config: {trials_per_config}")
    print(f"Total individual registrations to process: {total_runs}\n")
    
    # 2. Iterate through all parameter combinations
    with tqdm(total=total_configs, desc="Evaluating Configs") as pbar:
        for n_bits in N_list:
            for bags in bags_list:
                for tol in tolerance_list:
                    
                    config_reg_times = []
                    config_auth_times = []
                    successful_auth_count = 0

                    test_user_key = random.choice(user_keys)
                    face_base = mean_faces(facial_data[test_user_key])
                    
                    # --- Run 10 independent trials for this configuration ---
                    for trial in range(trials_per_config):
                        
                        # 3. Setup & Registration
                        server = Server(tolerance=tol, disjoint_bagging=bags)
                        
                        start_reg = time.time()
                        server.register("test_user", face_base, hyperplane=n_bits)
                        end_reg = time.time()
                        config_reg_times.append(end_reg - start_reg)
                        
                        # 4. Authentication (Up to 50 attempts)
                        for attempt in range(1, 51):
                            # Pick a random face sample
                            face_authen = random.choice(facial_data[test_user_key])
                            client2 = Client("test_user", face_authen)
                            
                            start_auth = time.time()
                            is_authenticated = server.authenticate("test_user", client2)
                            end_auth = time.time()
                            
                            if is_authenticated:
                                # Record ONLY the time of the successful run
                                config_auth_times.append(end_auth - start_auth)
                                successful_auth_count += 1
                                break # Move on to the next trial
                    
                    # --- Calculate Averages for this Configuration ---
                    avg_reg_time = sum(config_reg_times) / trials_per_config
                    
                    if successful_auth_count > 0:
                        avg_auth_time = sum(config_auth_times) / successful_auth_count
                    else:
                        avg_auth_time = "N/A"
                        
                    success_rate = successful_auth_count / trials_per_config
                    
                    # 5. Store the aggregated results
                    results.append({
                        'N_Hyperplanes': n_bits,
                        'Bags': bags,
                        'Tolerance': tol,
                        'Avg_Registration_Time_s': avg_reg_time,
                        'Avg_Authentication_Time_s': avg_auth_time,
                        'Auth_Success_Rate': success_rate,
                        'Successful_Trials': successful_auth_count
                    })
                    
                    # Update progress bar
                    pbar.set_postfix({
                        'Avg Reg': f"{avg_reg_time:.2f}s", 
                        'Success Rate': f"{success_rate*100:.0f}%"
                    })
                    pbar.update(1)

    # 6. Save and Return
    df_results = pd.DataFrame(results)
    output_filename = "safpake_robust_averages.csv"
    df_results.to_csv(output_filename, index=False)
    
    print(f"\n=== Experiment Complete! ===")
    print(f"Results successfully saved to: {output_filename}")
    
    return df_results

# --- Execute the Experiment ---
if __name__ == "__main__":
    # Ensure facial_data and mean_faces are available
    # df_robust = run_safpake_robust_experiment(facial_data, trials_per_config=10)
    pass

In [ ]:
df_res=run_safpake_experiment(facial_data)